### Building and Reinforcement learning agent for CarRacing 

In [11]:
## library installations 
! apt install swig cmake
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay
!pip install stable-baselines3 "gymnasium[box2d]"
! pip install huggingface_sb3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.2.0-2ubuntu1).
cmake is already the newest version (3.28.3-1build7).
0 upgraded, 0 newly installed, 0 to remove and 181 not upgraded.
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpa

In [12]:
## importing libraries 
import gymnasium # for lunar landing env 

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

# for rl algorithm library 
from stable_baselines3 import PPO # policy learning 
from stable_baselines3.common.env_util import make_vec_env # vectorized env 
from stable_baselines3.common.evaluation import evaluate_policy # policy evaluation 
from stable_baselines3.common.monitor import Monitor # monitoring env 

In [13]:
## basic iteration over the einvoronment 
import gymnasium as gym 

env = gym.make("CarRacing-v3",render_mode="rgb_array",continuous=True) # to get more precise angent (continuous = true)

observation,info = env.reset()
# print(observation,info)
for _ in range(20):
    action = env.action_space.sample() # taking random action [Stearing,gas,breaking]
    print("Action Taken:",action)
    observation,reward,terminated,truncated,info = env.step(action) # for tha action we get lhs
    if terminated or truncated:
        print("Environment is reset")
        observation,info = env.reset()

env.close() # important 

Action Taken: [-0.84908694  0.4829117   0.22648951]
Action Taken: [-0.862347    0.39498106  0.7091982 ]
Action Taken: [-0.29378223  0.75223565  0.1456203 ]
Action Taken: [0.29909468 0.20192508 0.98182887]
Action Taken: [-0.659015    0.42871472  0.78664875]
Action Taken: [0.8827901  0.5015237  0.50552773]
Action Taken: [0.83001846 0.11473785 0.67184514]
Action Taken: [-0.231517   0.6976388  0.865488 ]
Action Taken: [0.976115   0.26644817 0.26194325]
Action Taken: [-0.37787718  0.10092402  0.07196949]
Action Taken: [0.68634415 0.5252867  0.6466531 ]
Action Taken: [0.65069604 0.7539298  0.64001787]
Action Taken: [-0.30035752  0.68252665  0.48839572]
Action Taken: [-0.44509402  0.38842136  0.0732891 ]
Action Taken: [-0.21354857  0.20797813  0.74168074]
Action Taken: [-0.71891475  0.9695098   0.20207526]
Action Taken: [0.31381857 0.24322818 0.68814623]
Action Taken: [0.6555717  0.51456726 0.8834896 ]
Action Taken: [-0.6513769   0.9142106   0.13710214]
Action Taken: [-0.28917018  0.8082365  

In [14]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("CarRacing-v3",render_mode="rgb_array",continuous=True) # to get more infor here 
# A top-down 96x96 RGB image of the car and race track.
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Shape (96, 96, 3)
Sample observation [[[ 48 125 105]
  [ 33 119  93]
  [230 166 206]
  ...
  [ 19 187  32]
  [ 79 226  39]
  [167  39 227]]

 [[246 236 203]
  [195 255 107]
  [ 80  35  79]
  ...
  [143   8  99]
  [ 77 207 192]
  [220 189 227]]

 [[250 235 124]
  [226 105  44]
  [210  80 180]
  ...
  [188 222 113]
  [ 94 220  98]
  [207 245  67]]

 ...

 [[119   1 235]
  [118 101 229]
  [  3 148 130]
  ...
  [ 59  13 121]
  [207 111 155]
  [148 145 111]]

 [[251 208  38]
  [233 252 150]
  [172  62   7]
  ...
  [ 35 179  42]
  [160 149  51]
  [ 23 194 217]]

 [[ 75  52 210]
  [253 118  32]
  [236 242 229]
  ...
  [106  41 156]
  [ 98  38 151]
  [115 193 143]]]


In [15]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space) # [sterring,gas,break]
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape Box([-1.  0.  0.], 1.0, (3,), float32)
Action Space Sample [0.48859778 0.7927096  0.8031884 ]



#### Vectorized Environment
##### We create a vectorized environment (a method for stacking multiple independent environments into a single environment) of 16 environments, this way, ***we'll have more diverse experiences during the training.***

In [16]:
# Create the environment
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
## SubprocVecEnv is a class used to run multiple Gymnasium environments in parallel across separate CPU processes, bypassing Python's Global Interpreter Lock (GIL).
from stable_baselines3.common.vec_env import VecFrameStack
vec_env = make_vec_env(
    "CarRacing-v3",
    n_envs=16,
    seed=42,
    vec_env_cls=SubprocVecEnv,
    env_kwargs={
        "continuous": True,
        "domain_randomize": False,
        "lap_complete_percent": 0.95,
        "render_mode": "rgb_array",
    },
)
vec_env = VecFrameStack(vec_env, n_stack=4)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [17]:
## creating model and adding some params to accelerate training 
model = PPO(
    policy = "CnnPolicy", # because it is and image and depending on that the model is giving new policy(value) and action
    ## and ppo is basicially does the new policy should be not too far from the old policy. For that, ppo uses clipping to avoid too large update. , and that the ppo 
    env = vec_env, # parallel env 
    n_steps = 1024,# the number of steps each environment run before update 
    batch_size = 128, # min batch 
    n_epochs = 10, # Number of epoch when optimizing the surrogate loss
    gamma = 0.99, # discount factor to long term lerning 
    gae_lambda = 0.95, # Factor for trade-off of bias vs variance for Generalized Advantage Estimator
    ent_coef = 0.005, # entorpy coefficent for loss calculation. Its primialry use as the tradeoff between exploration and exploitation 
    verbose = 1,
    learning_rate=1e-4,
)

Using cuda device
Wrapping the env in a VecTransposeImage.


In [18]:
## train and ppo agent 
model.learn(total_timesteps=1000000)
## here each split is splitted in three main sections 
# 1. Rollout - environment interraction 
# 2. Time - Hardware & training speed
# 3.Train - Optimization & ppo losses  
## saving the model 
model_name = "carRacing-v3"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -57.7    |
| time/              |          |
|    fps             | 177      |
|    iterations      | 1        |
|    time_elapsed    | 92       |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -53.3       |
| time/                   |             |
|    fps                  | 163         |
|    iterations           | 2           |
|    time_elapsed         | 200         |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.005650663 |
|    clip_fraction        | 0.0649      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.23       |
|    explained_variance   | 0.022       |
|    learning_rate        | 0.

In [22]:
## Evaluate the model/agent 
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

base_eval_env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=True)
monitored_eval_env = Monitor(base_eval_env, filename="/kaggle/working/car_racing_log")

## 2. Vectorize the environment (required for stable-baselines3 image wrappers)
eval_env = DummyVecEnv([lambda: monitored_eval_env])

## 3. Transpose channels from (96, 96, 3) to (3, 96, 96)
eval_env = VecTransposeImage(eval_env)

## 4. Stack 4 frames to achieve the expected input channel dimension of 12
eval_env = VecFrameStack(eval_env, n_stack=4)

## Evaluate the model/agent
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward: {mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward: 645.79 +/- 191.13


In [41]:
# huggingface_login 
notebook_login()
! git config --global credential.helper store 

In [43]:
## setting some config 
model_architecture = "PPO"
env_id = "CarRacing-v3"
repo_id = "RahulKate-173/carRacing-rl-agent"
commit_message = "Uploaded PPO CarRacing-v3 trained agent"
# Create the base environment wrapped with DummyVecEnv
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])

# Add the necessary image preprocessing wrappers
eval_env = VecTransposeImage(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)

In [44]:
package_to_hub(
    model_name = model_name,
    model = model,
    model_architecture = model_architecture,
    env_id = env_id,
    eval_env = eval_env,
    repo_id = repo_id,
    commit_message = commit_message
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /tmp/tmphb5_kto5/-step-0-to-step-1000.mp4


/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /tmp/tmphb5_kto5/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmphb5_kto5/-step-0-to-step-1000.mp4

✘ must be real number, not NoneType
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo RahulKate-173/carRacing-rl-agent to the Hugging Face
Hub


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/vec_env/vec_video_recorder.py:157: UserWarning: WARN: Unable to save last video! Did you call close()?
  logger.warn("Unable to save last video! Did you call close()?")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/RahulKate-173/carRacing-rl-agent/tree/main/


CommitInfo(commit_url='https://huggingface.co/RahulKate-173/carRacing-rl-agent/commit/cfa5dfcde94c106b93656079c1485cb57a27ae52', commit_message='Uploaded PPO CarRacing-v3 trained agent', commit_description='', oid='cfa5dfcde94c106b93656079c1485cb57a27ae52', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RahulKate-173/carRacing-rl-agent', endpoint='https://huggingface.co', repo_type='model', repo_id='RahulKate-173/carRacing-rl-agent'), pr_revision=None, pr_num=None)

In [45]:
## creating and performance video 
import gymnasium as gym
import imageio
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from huggingface_hub import HfApi

# 1. Create the base environment
base_env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=True)

# 2. Add the exact same preprocessing wrappers used in training
eval_env = DummyVecEnv([lambda: base_env])
eval_env = VecTransposeImage(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)

frames = []
obs = eval_env.reset()  # VecEnv reset returns just the observation array
done = False

while not done:
    # Capture the raw RGB image array from the base environment for the video
    frame = base_env.render() 
    frames.append(frame)
    
    # Predict the action using your wrapped observation
    action, states = model.predict(obs, deterministic=True)
    
    # VecEnv step returns 4 values (dones handles termination & truncation automatically)
    obs, reward, dones, info = eval_env.step(action)
    done = dones[0]

eval_env.close()

# save frames 
video_path = '/kaggle/working/replay.mp4'
imageio.mimsave(video_path, frames, fps=30)
print(f"Replay saved: {len(frames)} frames recorded!")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Replay saved: 1000 frames recorded!


In [47]:
api = HfApi()
api.upload_file(
    path_or_fileobj = "/kaggle/working/replay.mp4",
    path_in_repo = "replay.mp4",
    repo_id = "Rahulkate-173/carRacing-rl-agent",
    repo_type = "model",
    commit_message = "replay video added"
)

print("Uploaded replay.mp4 successfully !!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded replay.mp4 successfully !!
